<a href="https://colab.research.google.com/github/VeX0niX/ML-Projects/blob/main/TBank_HW1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib
print(pd.__version__, np.__version__, sklearn.__version__, matplotlib.__version__)

2.2.3 2.1.3 1.6.1 3.10.0


In [ ]:
for enc in ['utf-8', 'cp1251', 'latin-1']:
    try:
        df = pd.read_csv('key.csv', encoding=enc)
        print("Получилось с кодировкой:", enc)
        break
    except Exception as e:
        print("Не вышло с", enc, "->", type(e).__name__)

print(df.shape)
df.head(10)

Не вышло с utf-8 -> ParserError
Не вышло с cp1251 -> UnicodeDecodeError
Не вышло с latin-1 -> ParserError
(2128, 4)


,pos,shift,relay,shift_num
0,0,NaN,LaTrappe-42,NaN
1,1,121,LaTrappe-42,121.0
2,2,83,LaTrappe-42,83.0
3,3,174,LaTrappe-42,174.0
4,4,NaN,LaTrappe-42,NaN
5,5,115,LaTrappe-42,115.0
6,6,83,LaTrappe-42,83.0
7,7,106,LaTrappe-42,106.0
8,8,127,LaTrappe-42,127.0
9,9,119,LaTrappe-42,119.0


In [ ]:
# shift в числа, всё нечисловое в NaN
df['shift_num'] = pd.to_numeric(df['shift'], errors='coerce')

# строки = pos, столбцы = relay
pivot = df.pivot(index='pos', columns='relay', values='shift_num')

print("Размер таблицы:", pivot.shape)
print("Список ретрансляторов:")
for col in pivot.columns:
    print("  ", col)

Размер таблицы: (152, 14)
Список ретрансляторов:
   Achel-19
   Chimay-23
   Engelszell-28
   Koningshoeven-05
   LaTrappe-42
   MontDesCats-77
   Orval-07
   Rochefort-10
   Spencer-61
   TreFontane-33
   TyntMeadow-88
   Westmalle-31
   Westvleteren-12
   Zundert-56


In [ ]:
for relay in ['Spencer-61', 'MontDesCats-77', 'Orval-07', 'Achel-19']:
    col = pivot[relay]
    print(f"\n=== {relay} ===")
    print("Непустых значений:", col.notna().sum(), "из", len(col))
    print("Первые 12 значений:", col.head(12).tolist())


=== Spencer-61 ===
Непустых значений: 0 из 152
Первые 12 значений: [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]

=== MontDesCats-77 ===
Непустых значений: 152 из 152
Первые 12 значений: [176.0, 176.0, 176.0, 176.0, 176.0, 176.0, 176.0, 176.0, 176.0, 176.0, 176.0, 176.0]

=== Orval-07 ===
Непустых значений: 152 из 152
Первые 12 значений: [94.0, 34.0, 196.0, 140.0, 94.0, 34.0, 196.0, 140.0, 94.0, 34.0, 196.0, 140.0]

=== Achel-19 ===
Непустых значений: 100 из 152
Первые 12 значений: [130.0, 121.0, 83.0, nan, nan, nan, 83.0, 58.0, nan, 185.0, nan, nan]


In [ ]:
# Исключаем заведомо плохие
bad = ['Spencer-61', 'MontDesCats-77', 'Orval-07', 'Achel-19']
candidates = pivot.drop(columns=bad)

# Мода каждой позиции
def mode(series):
    s = series.dropna()
    return s.mode().iloc[0] if len(s) else None

majority = candidates.apply(mode, axis=1)

# Считаем расхождения каждой станции с большинством
disagreements = {}
for col in candidates.columns:
    mask = candidates[col].notna() & majority.notna()
    disagreements[col] = (candidates[col][mask] != majority[mask]).sum()

# Сортируем по числу расхождений
import pandas as pd
print(pd.Series(disagreements).sort_values(ascending=False))

Chimay-23           99
Westmalle-31        85
Engelszell-28       22
LaTrappe-42         20
TyntMeadow-88       16
TreFontane-33       15
Zundert-56          15
Koningshoeven-05    13
Westvleteren-12     13
Rochefort-10        12
dtype: int64


In [ ]:
# Полный список плохих
bad = ['Spencer-61', 'MontDesCats-77', 'Orval-07', 'Achel-19',
       'Chimay-23', 'Westmalle-31']

honest = pivot.drop(columns=bad)
print("Осталось станций:", honest.shape[1])
print(list(honest.columns))

# Ключ = мода по честным станциям
key = honest.apply(mode, axis=1)

print("\nПозиций с NaN в ключе:", key.isna().sum())
print("Первые 20 значений ключа:", key.head(20).tolist())

Осталось станций: 8
['Engelszell-28', 'Koningshoeven-05', 'LaTrappe-42', 'Rochefort-10', 'TreFontane-33', 'TyntMeadow-88', 'Westvleteren-12', 'Zundert-56']

Позиций с NaN в ключе: 0
Первые 20 значений ключа: [130.0, 121.0, 83.0, 174.0, 77.0, 115.0, 83.0, 31.0, 127.0, 119.0, 146.0, 158.0, 206.0, 43.0, 39.0, 151.0, 133.0, 52.0, 111.0, 159.0]


In [ ]:
import hashlib

key = key.astype(int)

total_sum = key.sum()
print("Сумма всех сдвигов ключа:", total_sum)

key_list = key.tolist()
hash_str = hashlib.sha256(",".join(map(str, key_list)).encode()).hexdigest()
expected = "08a09df10bbe00b15d9fbe7f0c2937cd3e6986aa920adc7fab83d762c443ecfd"
print("Вычисленный хеш:", hash_str)
print("Ожидаемый хеш:  ", expected)
print("Совпадает:", hash_str == expected)

Сумма всех сдвигов ключа: 16024
Вычисленный хеш: 08a09df10bbe00b15d9fbe7f0c2937cd3e6986aa920adc7fab83d762c443ecfd
Ожидаемый хеш:   08a09df10bbe00b15d9fbe7f0c2937cd3e6986aa920adc7fab83d762c443ecfd
Совпадает: True


In [ ]:
codebook = pd.read_csv('codebook.csv')
print("Размер алфавита N =", len(codebook))
print(codebook.head())

Размер алфавита N = 211
   code char
0    97    а
1    11    б
2    27    в
3    48    г
4    45    д


In [ ]:
code_to_char = dict(zip(codebook['code'], codebook['char']))
cipher = pd.read_csv('message.csv').sort_values('pos')
plain = ''.join(code_to_char[(int(r['code']) - key[int(r['pos'])]) % 211]
                for _, r in cipher.iterrows())
print(plain)

Привет, друг! Жители TRAPPIST-1d поздравляют тебя с началом занятий. Per aspera ad astra! У тебя все получится. Код посадочной площадки: 86W-V4W-H75-LK9


In [ ]:
col = df[df['relay'] == 'Achel-19']['shift']

nonempty = col.notna()

not_numbers = pd.to_numeric(col, errors='coerce').isna() & nonempty
print(not_numbers.sum())

26


In [ ]:
print(col[not_numbers].tolist())

['1e', '…', 'ехи', 'sig', 'ехи', '@@', '--', 'ERR', 'sig', '☼', '1e', '…', 'sig', 'ok', '…', 'lost', '2026-09-13', 'sig', '12:40', '--', '12:40', 'ехи', '2026-09-13', 'lost', '13.09.2026', '--']
